<a href="https://colab.research.google.com/github/ac1d301/clevercsv_speedup/blob/main/baseline_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!cat /proc/cpuinfo | grep "model name" | head -1
!free -h
!python --version

model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
               total        used        free      shared  buff/cache   available
Mem:            12Gi       604Mi       9.1Gi       1.0Mi       3.0Gi        11Gi
Swap:             0B          0B          0B
Python 3.12.13


In [3]:
!git clone https://github.com/ac1d301/clevercsv_speedup.git
!pip install clevercsv==0.8.3 -q
import clevercsv
print(clevercsv.__version__)

Cloning into 'clevercsv_speedup'...
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 11 (delta 1), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (11/11), 4.25 MiB | 9.80 MiB/s, done.
Resolving deltas: 100% (1/1), done.
0.8.3


In [11]:
import os

data_path = "/content/clevercsv_speedup/data/fec_sample.txt"
print(f"File exists: {os.path.exists(data_path)}")

with open(data_path, encoding='latin-1') as f:
    raw = f.read()

lines = raw.split('\n')
data = '\n'.join(lines[:25000])

print(f"Total rows available: {len(lines)}")
print(f"Using rows: {len(data.split(chr(10)))}")
print(f"Data size: {len(data)/1e6:.2f} MB")
print(f"Sample row: {lines[1][:120]}")

File exists: True
Total rows available: 120001
Using rows: 25000
Data size: 4.81 MB
Sample row: C00580100|A|YE|P2020|201903139145683413|15|IND|GEARY, ESTHER|HUNTINGTON BEACH|CA|92649|RETIRED|RETIRED|12312018|250||SA1


In [12]:
import time, statistics

# 2 warmup runs (not measured)
print("Warming up...")
for i in range(2):
    t0 = time.perf_counter()
    dialect = clevercsv.Detector().detect(data)
    print(f"  Warmup {i+1}: {time.perf_counter()-t0:.2f}s")

# 5 measured runs
print("\nMeasuring...")
times = []
for i in range(5):
    t0 = time.perf_counter()
    dialect = clevercsv.Detector().detect(data)
    elapsed = time.perf_counter() - t0
    times.append(elapsed)
    print(f"  Run {i+1}: {elapsed:.2f}s")

times_sorted = sorted(times)
median = statistics.median(times)
iqr = times_sorted[3] - times_sorted[1]

print(f"\nMedian: {median:.2f}s")
print(f"IQR:    {iqr:.2f}s")
print(f"Dialect detected: {dialect}")

Warming up...
  Warmup 1: 22.11s
  Warmup 2: 20.26s

Measuring...
  Run 1: 19.96s
  Run 2: 22.49s
  Run 3: 19.96s
  Run 4: 22.28s
  Run 5: 20.30s

Median: 20.30s
IQR:    2.32s
Dialect detected: SimpleDialect('|', '', '')


In [13]:
import pickle

golden = {
    'dialect_delimiter': dialect.delimiter,
    'dialect_quotechar': dialect.quotechar,
    'dialect_escapechar': dialect.escapechar,
    'median_time': 20.30,
    'iqr': 2.32,
    'n_rows': 25000
}

with open('baseline_output.pkl', 'wb') as f:
    pickle.dump(golden, f)

print("Golden output saved to baseline_output.pkl")
print(golden)

Golden output saved to baseline_output.pkl
{'dialect_delimiter': '|', 'dialect_quotechar': '', 'dialect_escapechar': '', 'median_time': 20.3, 'iqr': 2.32, 'n_rows': 25000}


In [14]:
!git clone https://github.com/alan-turing-institute/CleverCSV.git
%cd CleverCSV
!git checkout ae043c948fd03eea2ae726525c4f347646d22316
!pip install -e ".[dev]" -q
!python -m pytest tests/ -v --tb=short 2>&1 | tee pytest_baseline.txt

Cloning into 'CleverCSV'...
remote: Enumerating objects: 4623, done.
remote: Counting objects: 100% (456/456), done.
remote: Compressing objects: 100% (193/193), done.
remote: Total 4623 (delta 297), reused 298 (delta 262), pack-reused 4167 (from 2)
Receiving objects: 100% (4623/4623), 3.68 MiB | 16.31 MiB/s, done.
Resolving deltas: 100% (3032/3032), done.
/content/CleverCSV
Note: switching to 'ae043c948fd03eea2ae726525c4f347646d22316'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at ae043c9 CleverCSV Release 0

In [15]:
import json, re

with open('pytest_baseline.txt') as f:
    output = f.read()

# Extract passed test names
passed = re.findall(r'PASSED (.+)', output)
failed = re.findall(r'FAILED (.+)', output)

test_results = {
    'passed': passed,
    'failed': failed,
    'passed_count': len(passed),
    'failed_count': len(failed)
}

with open('baseline_test_results.json', 'w') as f:
    json.dump(test_results, f, indent=2)

print(f"Passed: {len(passed)}")
print(f"Failed: {len(failed)}")

Passed: 164
Failed: 0
